# Grad-CAM da U-Net (estágio 13)

Gera mapas de calor **Grad-CAM** para a **U-Net (CNN)** usando os pesos treinados no estágio 09 e os artefatos de avaliação do estágio 11 da **fonte escolhida manualmente** (`ground_truth.chosen_source`). Para patches representativos da validação espacial, a camada-alvo é o bloco do codificador mais profundo (`encoders[-1]`) e os gradientes são agregados pela soma dos logits da classe café na região de referência do patch. Os mapas são persistidos em `MyDrive/tcc/artifacts/xai/gradcam/unet/<fonte>/` e visualizados em uma figura com RGB, referência, Grad-CAM e sobreposição. O estágio não retreina: apenas explica os pesos existentes, usando o modelo da dobra de cada patch.


## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.


In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc-lab/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## Dependências pinadas

Garante as versões fixadas em `requirements-runtime.txt`, o mesmo conjunto nas duas plataformas. Quando uma biblioteca com extensão compilada já importada (numpy, scipy, torch, entre outras) diverge da versão pinada, o kernel é reiniciado automaticamente para evitar inconsistência de ABI; nesse caso, o notebook é reexecutado a partir da primeira célula.


In [ ]:
# Garante as versões pinadas e reinicia o kernel quando a ABI carregada divergir.
from src.setup import install_runtime_requirements

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
install_runtime_requirements(requirements)


## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo modelos, artefatos de avaliação, mapas de XAI e figuras. A fonte explicada é a mesma escolhida manualmente na fase de ground truth.


In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src.data.source_decision import chosen_source
from src.setup import initialize
from src.xai.gradcam import gradcam_dir, gradcam_figure_path, gradcam_maps_path

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
source_name = chosen_source()
print(f"Plataforma: {platform}")
print(f"Fonte de treinamento/avaliação (decisão manual): {source_name}")
print(f"Mapas Grad-CAM: {gradcam_dir(storage_paths, source_name)}")
print(f"Arquivo de mapas: {gradcam_maps_path(storage_paths, source_name)}")
print(f"Figura: {gradcam_figure_path(storage_paths, source_name)}")


## Reprodutibilidade e dispositivo

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões do ambiente. O Grad-CAM usa autograd apenas para obter os gradientes da camada-alvo, sem atualizar pesos. O processamento ocorre na GPU quando disponível e recai para CPU quando não há acelerador — viável porque apenas poucos patches representativos são explicados.


In [ ]:
# Fixa sementes, flags determinísticas e seleciona o dispositivo de execução.
import torch

from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"Dispositivo: {device} | {torch.cuda.get_device_name(0)}")
else:
    print(f"Dispositivo: {device} (GPU indisponível; Grad-CAM em CPU)")


## Dependências dos estágios 06, 08, 09 e 11

Verifica que o manifesto (estágio 06), as estatísticas de normalização (estágio 08), os pesos e artefatos de treino da U-Net (estágio 09) e o resumo vigente da avaliação (estágio 11) estão disponíveis. A explicação usa o modelo da dobra de cada patch selecionado, sem reavaliar todos os patches.


In [ ]:
# Verifica as dependências dos estágios anteriores e a vigência dos artefatos da U-Net.
from src.data.eda import normalization_stats_is_current, normalization_stats_path
from src.data.patch_generation import manifest_is_current, manifest_path
from src.data.spatial_split import fold_count, split_is_current
from src.evaluation import evaluation_summary_path, summary_is_current
from src.trainer import fold_training_done
from src.utils import check_dependencies

check_dependencies(
    {
        "Manifesto (estágio 06)": manifest_path(storage_paths, source_name),
        "Estatísticas de normalização (estágio 08)": normalization_stats_path(
            storage_paths, source_name
        ),
        "Resumo da avaliação (estágio 11)": evaluation_summary_path(
            storage_paths, "unet", source_name
        ),
    }
)
if not manifest_is_current(storage_paths, source_name):
    raise ValueError("Manifesto desatualizado frente às entradas; execute o estágio 06 antes.")
if not split_is_current(storage_paths, source_name):
    raise ValueError("Divisão espacial desatualizada; execute o estágio 07 antes.")
if not normalization_stats_is_current(storage_paths, source_name):
    raise ValueError("Estatísticas de normalização desatualizadas; execute o estágio 08 antes.")
if not summary_is_current(storage_paths, "unet", source_name):
    raise ValueError("Resumo da U-Net desatualizado; reexecute o estágio 11 antes do Grad-CAM.")
for fold in range(fold_count()):
    if not fold_training_done(storage_paths, "unet", source_name, fold):
        raise FileNotFoundError(
            f"Pesos/artefatos de treino ausentes para a U-Net, dobra {fold}; "
            "execute o estágio 09 antes do Grad-CAM."
        )
print(f"Dobras da U-Net verificadas: {fold_count()}")


## Geração dos mapas Grad-CAM

Seleciona de forma determinística patches representativos com café (maior e menor IoU, maior proporção de café e IoU mediano), carrega a imagem normalizada e a máscara de referência, e calcula o Grad-CAM com os pesos da dobra de cada patch. Os mapas, o metadata com fingerprint e a figura são persistidos no Drive; a geração é reutilizada apenas enquanto o resumo do estágio 11, os pesos das dobras e a configuração permanecem vigentes.


In [ ]:
# Gera (ou reutiliza) os mapas Grad-CAM da U-Net na fonte escolhida.
from src.xai.gradcam import run_gradcam

gradcam_result = run_gradcam(storage_paths, source_name=source_name, device=device)
print(f"Artefatos reutilizados: {gradcam_result.skipped}")
print(f"Patches explicados: {gradcam_result.metadata['n_patches']}")


## Patches explicados e métricas

Apresenta os patches selecionados, a dobra que forneceu os pesos, o IoU da U-Net, a proporção de café e o hash dos pesos usados em cada mapa, permitindo rastrear exatamente qual modelo gerou cada explicação.


In [ ]:
# Monta a tabela dos patches explicados e dos pesos utilizados.
import pandas as pd

pd.DataFrame(
    [
        {
            "patch_id": item["patch_id"],
            "caso": item["label"],
            "dobra": item["fold"],
            "iou": round(item["iou"], 4),
            "coffee_ratio": round(item["coffee_ratio"], 4),
            "weights_sha256": item["weights_sha256"],
        }
        for item in gradcam_result.metadata["patches"]
    ]
)


## Resumo da etapa

Exibe o resumo do estágio com a fonte, a camada-alvo, a região de pontuação, o número de patches explicados e os caminhos dos mapas, do metadata e da figura.


In [ ]:
# Exibe o resumo da etapa e os caminhos dos artefatos de XAI.
from src.utils import print_summary

summary = {
    "Fonte": source_name,
    "Modelo": gradcam_result.metadata["model"],
    "Camada alvo": gradcam_result.metadata["target_layer"],
    "Região de pontuação": gradcam_result.metadata["score_region"],
    "Patches explicados": gradcam_result.metadata["n_patches"],
    "Dispositivo": gradcam_result.metadata["device"],
    "Mapas": str(gradcam_result.maps_path),
    "Metadata": str(gradcam_result.metadata_path),
    "Figura": str(gradcam_result.figure_path),
}
print_summary(summary, "13")
